In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score,f1_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
df = pd.read_csv("fake_news_dataset.csv")

In [ ]:
#missing value in datsets
df.isnull().sum()

,0
title,0
text,0
date,1
source,697
author,719
category,1
label,1


In [ ]:
#replacing null values with empty string
df = df.fillna('')

In [ ]:
#merging the auther name news title
df['content'] = df['author']+' '+df['title']

In [ ]:
print(df['content'])

0                     Paula George Foreign Democrat final.
1          Joseph Hill To offer down resource great point.
2              Julia Robinson Himself church myself carry.
3                Mr. David Foster DDS You unit its should.
4        Austin Walker Billion believe employee summer ...
                               ...                        
14014               Duane Sloan Rather there but this his.
14015    Candice Adams Analysis some people use small all.
14016    Kaylee Jensen Win million base daughter list f...
14017    Gloria Anderson Test cell possible audience very.
14018                                  Laugh cover season.
Name: content, Length: 14019, dtype: object


In [ ]:
#separating the data  and label
X = df.drop('label' , axis =1)
y=df['label']
X.head()

,title,text,date,source,author,category,content
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,Paula George Foreign Democrat final.
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,Joseph Hill To offer down resource great point.
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,Julia Robinson Himself church myself carry.
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,Mr. David Foster DDS You unit its should.
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,Austin Walker Billion believe employee summer ...


In [ ]:


port_stem = PorterStemmer()

def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()

    stemmed_content = [
        port_stem.stem(word)
        for word in stemmed_content
        if word not in stopwords.words('english')
    ]

    stemmed_content = ' '.join(stemmed_content)

    return stemmed_content

In [ ]:
df['content' ] = df['content'].apply(stemming)

In [ ]:
print(df['content'])

0                       paula georg foreign democrat final
1                    joseph hill offer resourc great point
2                              julia robinson church carri
3                                  mr david foster dd unit
4              austin walker billion believ employe summer
                               ...                        
14014                                    duan sloan rather
14015                  candic adam analysi peopl use small
14016    kayle jensen win million base daughter list fi...
14017            gloria anderson test cell possibl audienc
14018                                   laugh cover season
Name: content, Length: 14019, dtype: object


In [ ]:
# Separate data and label

X = df['content'].values

y = df['label'].astype(str).str.strip().str.lower()

# Convert labels
y = y.map({
    "fake": 0,
    "real": 1
})

# Remove rows where label is NaN
valid = y.notna()

X = X[valid]
y = y[valid]

# Convert y to integer
y = y.astype(int).values


# Convert text into numerical data

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(X)


# Split data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

In [ ]:
#Evaluation
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
print("accuracy training :", accuracy_score(y_train_pred,y_train))
print("accuracy testing:", accuracy_score(y_test_pred,y_test))

accuracy training : 0.6713037274835028
accuracy testing: 0.5121255349500713


In [ ]:
X_new = X_test[0]
prediction = model.predict(X_new)
print(prediction)

if(prediction[0]==[0]):
  print('The news is Real')

else:
  print('The news is Fake')

[1]
The news is Fake
